# Ryan PINN: one model per cell
Run cells sequentially in one GPU runtime. Set the reviewed commit SHA below before setup. Each training cell executes one frozen 8,000-update model and displays training progress. You may pause between completed cells. All six primary models must terminate before primary evaluation; stress execution follows primary evaluation. The notebook only launches version-controlled repository scripts.

If protected attempts already exist at a different Git SHA, retain that original revision. Never mix revisions, erase attempts, or retry a scientific failure.

In [ ]:
COMMIT_SHA = "PASTE_REVIEWED_COMMIT_SHA"
from google.colab import drive
from pathlib import Path
import re
assert re.fullmatch(r"[0-9a-f]{40}", COMMIT_SHA), "Paste the exact reviewed 40-character Git SHA"
drive.mount("/content/drive")
BASE = Path("/content/drive/MyDrive/ryan_pinn_trial")
REPO = BASE / "repository"

In [ ]:
import subprocess
import sys
assert Path("/content/drive/MyDrive").is_dir(), "Mount Drive first"
BASE.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/Arawal123/helmholtz-pinn-rff-benchmark.git", str(REPO)], check=True)
assert (REPO / ".git").is_dir(), "Preserve this folder and inspect it: not a Git checkout"
def git(*args):
    return subprocess.check_output(["git", *args], cwd=REPO, text=True).strip()
assert not git("status", "--porcelain"), "The reviewed checkout must be clean"
if git("rev-parse", "HEAD") != COMMIT_SHA and any((REPO / "results/raw").glob("n*/**/attempt_*")):
    raise RuntimeError("Protected attempts exist at another SHA. Continue using their original frozen revision.")
subprocess.run(["git", "fetch", "origin"], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", COMMIT_SHA], cwd=REPO, check=True)
assert git("rev-parse", "HEAD") == COMMIT_SHA
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=REPO, check=True)
print("Frozen commit:", COMMIT_SHA)

In [ ]:
from datetime import datetime, timezone
import shutil
import signal

def run_stage(*args):
    """Launch repository CLI and persist its console output; no scientific logic."""
    log = BASE / ("console_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".log")
    command = [sys.executable, "-u", "scripts/run_frozen.py", "--config", "configs/frozen.yaml", *args]
    print("Command:", " ".join(command), "\nLog:", log)
    with log.open("x", encoding="utf-8") as output:
        process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                output.write(line)
                output.flush()
            returncode = process.wait()
        except BaseException:
            if process.poll() is None:
                process.send_signal(signal.SIGINT)
                try:
                    process.wait(timeout=20)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
            raise
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

subprocess.run([sys.executable, "-c", "import torch; assert torch.cuda.is_available(), 'GPU unavailable'; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))"], cwd=REPO, check=True)
subprocess.run([sys.executable, "scripts/validate_protocol.py"], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, check=True)
(BASE / "code_sha.txt").write_text(COMMIT_SHA + "\n", encoding="utf-8")
with (BASE / "environment.txt").open("w", encoding="utf-8") as f:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=f, check=True)
if not (BASE / "EXPERIMENT_LOG_execution.md").exists():
    shutil.copy2(REPO / "EXPERIMENT_LOG.md", BASE / "EXPERIMENT_LOG_execution.md")
run_stage("--action", "status")

## n=6: primary condition
Run each training cell once. Repeating a terminal slot safely skips it. Preserve numerical failures.

**Train n=6, vanilla, seed 0.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "vanilla", "--seed", "0")

**Train n=6, vanilla, seed 1.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "vanilla", "--seed", "1")

**Train n=6, vanilla, seed 2.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "vanilla", "--seed", "2")

**Train n=6, rff, seed 0.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "rff", "--seed", "0")

**Train n=6, rff, seed 1.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "rff", "--seed", "1")

**Train n=6, rff, seed 2.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "6", "--method", "rff", "--seed", "2")

**Evaluate n=6.** The repository refuses this cell until all six training slots for this condition are terminal.

In [ ]:
run_stage("--action", "evaluate", "--n", "6")

## n=12: preregistered stress/failure condition
Run each training cell once. Repeating a terminal slot safely skips it. Preserve numerical failures.

**Train n=12, vanilla, seed 0.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "vanilla", "--seed", "0")

**Train n=12, vanilla, seed 1.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "vanilla", "--seed", "1")

**Train n=12, vanilla, seed 2.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "vanilla", "--seed", "2")

**Train n=12, rff, seed 0.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "rff", "--seed", "0")

**Train n=12, rff, seed 1.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "rff", "--seed", "1")

**Train n=12, rff, seed 2.** Stop here between cells if desired; the full model always retains its approved update budget.

In [ ]:
run_stage("--action", "train", "--n", "12", "--method", "rff", "--seed", "2")

**Evaluate n=12.** The repository refuses this cell until all six training slots for this condition are terminal.

In [ ]:
run_stage("--action", "evaluate", "--n", "12")

## Generate all deliverables
This cell requires all 12 training slots and both condition evaluations to be complete. It generates the five processed CSVs, figures, and methods/results note.

In [ ]:
run_stage("--action", "finalize")
run_stage("--action", "status")

## Package and download
Append observations, scientific failures, interruptions, and decisions to the external execution log. This archive contains source, frozen assets, raw/processed evidence, figures, report, SHA, environment, console logs, and journal.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive = BASE / ("ryan_pinn_submission_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".zip")
excluded = {".git", "__pycache__", ".pytest_cache", "smoke_outputs"}
with ZipFile(archive, "x", ZIP_DEFLATED) as z:
    for path in BASE.rglob("*"):
        if path.is_file() and path.suffix != ".zip" and not excluded.intersection(path.relative_to(BASE).parts):
            z.write(path, path.relative_to(BASE))
print("Saved:", archive)
files.download(str(archive))

## Infrastructure recovery only
After a runtime interruption, remount Drive and repeat setup at the same SHA. For the affected training or evaluation cell, add `"--recover-infrastructure"` to `run_stage(...)`. Prior attempts remain intact; interrupted training restarts from update 1 under the identical specification. Completed or scientifically failed models are never retrained. Run only one cell/process at a time. The `status` action reads training metadata and never displays held-out metrics.